## 🎬 Movie Recommender System

#### Project Overview

    This project focuses on building a **content-based movie recommendation system** using the TMDB 5000 Movies and Credits datasets.

    The system takes a movie title as input and recommends other movies based on the similarity of their available metadata and textual features.

    The recommendation pipeline includes:

    - Data cleaning and preprocessing
    - Feature selection and engineering
    - Combining movie metadata from two datasets
    - Extracting useful information from genres, keywords, cast and crew
    - Creating a combined `tags` feature
    - Text preprocessing and lemmatization
    - Text vectorization
    - Cosine similarity
    - Recommendation generation
    - Comparison of multiple modelling approaches
    - Selection of a final recommendation configuration

In [504]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [505]:
df_credits = pd.read_csv(r'C:\Users\moham\Desktop\Movie-Recommender-System\data\raw\tmdb_5000_credits.csv')
df_movies = pd.read_csv(r'C:\Users\moham\Desktop\Movie-Recommender-System\data\raw\tmdb_5000_movies.csv')
df_credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [506]:
df_movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


### 1. Dataset

    This project uses two datasets from the TMDB 5000 Movie Dataset:

    1. `tmdb_5000_movies.csv`
    2. `tmdb_5000_credits.csv`

    The two datasets contain complementary information about the movies.

    The movies dataset provides information such as:

    - Movie title
    - Original title
    - Overview
    - Genres
    - Keywords
    - Movie metadata

    The credits dataset provides information such as:

    - Cast
    - Crew
    - Movie ID

    The datasets are combined using the common movie identifier so that relevant information from both sources can be used for recommendation.

# Data Cleaning

In [507]:
(df_movies['title'] == df_movies['original_title']).all()

np.False_

In [508]:
df_movies[df_movies['title'] != df_movies['original_title']][['title','original_title']].head(10)

,title,original_title
97,Shin Godzilla,シン・ゴジラ
215,Fantastic 4: Rise of the Silver Surfer,4: Rise of the Silver Surfer
235,Asterix at the Olympic Games,Astérix aux Jeux Olympiques
317,The Flowers of War,金陵十三釵
474,Evolution,Évolution
488,Arthur and the Invisibles,Arthur et les Minimoys
492,Top Cat Begins,Don Gato: El inicio de la pandilla
561,Two Brothers,Deux frères
678,Dragon Blade,天將雄師
719,This Is It,Michael Jackson's This Is It


In [509]:
movies = df_movies[['id', 'title', 'original_title', 'overview', 'genres', 'keywords']]
movies.head()

,id,title,original_title,overview,genres,keywords
0,19995,Avatar,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
1,285,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
2,206647,Spectre,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name..."
3,49026,The Dark Knight Rises,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,..."
4,49529,John Carter,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":..."


In [510]:
credits = df_credits.drop(columns='title')
credits.head()

,movie_id,cast,crew
0,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [511]:
movies = movies.merge(
    credits,
    left_on='id',
    right_on='movie_id'
)
movies.head(3)

,id,title,original_title,overview,genres,keywords,movie_id,cast,crew
0,19995,Avatar,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [512]:
movies.drop(columns=['id','original_title'], inplace=True)

In [513]:
movies.duplicated().sum()

np.int64(0)

In [514]:
movies.isna().sum()

title       0
overview    3
genres      0
keywords    0
movie_id    0
cast        0
crew        0
dtype: int64

In [515]:
movies.dropna(inplace=True)

In [516]:
movies.head(3)

,title,overview,genres,keywords,movie_id,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [517]:
movies['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [518]:
import ast
def convert1(text):
    list_genres = []
    text = ast.literal_eval(text)
    for item in text:
        list_genres.append(item['name'])
    return list_genres

In [519]:
movies['genres'] = movies['genres'].apply(convert1)

In [520]:
movies['keywords'][0]

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [521]:
def convert2(text):
    list_kewords = []
    text = ast.literal_eval(text)
    for item in text:
        list_kewords.append(item['name'])
    return list_kewords

In [522]:
movies['keywords'] = movies['keywords'].apply(convert2)

In [523]:
def convert3(text):
    list_cast = []
    text = ast.literal_eval(text)
    count = 0
    for item in text:
        if count > 7:
            break
        else:
            list_cast.append(item['name'])
            count += 1

    return list_cast


In [524]:
movies['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [525]:
movies['cast'] = movies['cast'].apply(convert3)

In [526]:
movies['crew'][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [527]:
def convert4(text):
    list_director = []
    text = ast.literal_eval(text)
    for item in text:
        if item['job'] == 'Director':
            list_director.append(item['name'])

    return list_director

In [528]:
movies['crew'] = movies['crew'].apply(convert4)

In [529]:
movies['cast'] = movies['cast'].apply(lambda x : [item.strip().replace(" ","") for item in x])
movies['crew'] = movies['crew'].apply(lambda x : [item.strip().replace(" ","") for item in x])
movies['keywords'] = movies['keywords'].apply(lambda x : [item.strip().replace(" ","") for item in x])
movies['genres'] = movies['genres'].apply(lambda x : [item.strip().replace(" ","") for item in x])

In [530]:
movies['genres'] = movies['genres'].apply(lambda x : " ".join(x))
movies['keywords'] = movies['keywords'].apply(lambda x : " ".join(x))
movies['crew'] = movies['crew'].apply(lambda x : " ".join(x))
movies['cast'] = movies['cast'].apply(lambda x : " ".join(x))

In [531]:
movies['tags'] = (
    movies['overview'] + ' ' +
    movies['genres'] + ' ' +
    movies['keywords'] + ' ' +
    movies['cast'] + ' ' +
    movies['crew']
)

In [532]:
new_movies = movies[['movie_id','title','tags']]

In [533]:
new_movies['tags'] = new_movies['tags'].apply(lambda x : x.lower())

## Model 1 :  BOW Without Using Movie Title Column

In [534]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors1 = cv.fit_transform(new_movies['tags'])

In [535]:
vectors1

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 138585 stored elements and shape (4800, 5000)>

In [536]:
cv.get_feature_names_out()[:90]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950s', '1960s',
       '1970s', '1980', '1980s', '1985', '1990s', '19th', '19thcentury',
       '20', '200', '2009', '20th', '24', '25', '30', '300', '3d', '40',
       '50', '500', '60', '60s', '70', 'aaron', 'aaroneckhart',
       'aarontaylor', 'abandoned', 'abbiecornish', 'abducted',
       'abigailbreslin', 'abilities', 'ability', 'able', 'aboard',
       'abuse', 'abusive', 'academy', 'accept', 'accepted', 'accepts',
       'access', 'accident', 'accidental', 'accidentally', 'accompanied',
       'accomplish', 'account', 'accountant', 'accused', 'ace', 'achieve',
       'act', 'acting', 'action', 'actionhero', 'actions', 'activist',
       'activities', 'activity', 'actor', 'actors', 'actress', 'acts',
       'actual', 'actually', 'adam', 'adambrody', 'adamgoldberg', 'adams',
       'adamsandler', 'adamscott', 'adamshankman', 'adaptation'],
      dtype=object

In [537]:
import nltk

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-d

True

#### Why Lemmatization?
    Different grammatical forms of the same word may otherwise become separate features.
    For example:
    `act`, `acting`, `acted`
    represent closely related concepts.
    Lemmatization attempts to convert these different forms into a common dictionary-based form.
    This can reduce unnecessary vocabulary variation and improve the consistency of text representation.

In [538]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):

    tag = pos_tag([word])[0][1]
    if tag.startswith('J'):
        return wordnet.ADJ
    
    elif tag.startswith('V'):
        return wordnet.VERB
    
    elif tag.startswith('N'):
        return wordnet.NOUN
    
    elif tag.startswith('R'):
        return wordnet.ADV
    
    else:
        return wordnet.NOUN


def lemmatize_text(text):
    
    words = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))


    filtered_word = []
    for word in words:
        if word not in stop_words:
            filtered_word.append(word)


    lemmatized_words = []
    for word in filtered_word:
        pos = get_wordnet_pos(word)
        lemma = lemmatizer.lemmatize(word,pos)
        lemmatized_words.append(lemma)

  
    return " ".join(lemmatized_words)

In [539]:
new_movies['tags'] = new_movies['tags'].apply(lemmatize_text)

In [540]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors1 = cv.fit_transform(new_movies['tags'])

In [541]:
cv.get_feature_names_out()[:90]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950s', '1960s',
       '1970s', '1980s', '1985', '1990s', '19th', '19thcentury', '20',
       '200', '2009', '20th', '24', '25', '30', '300', '3d', '40', '50',
       '500', '60', '70', 'aaron', 'aaroneckhart', 'aarontaylor',
       'abandon', 'abbiecornish', 'abduct', 'abigailbreslin', 'ability',
       'able', 'aboard', 'abuse', 'abusive', 'academy', 'accept',
       'accepted', 'accepts', 'access', 'accident', 'accidental',
       'accidentally', 'acclaim', 'accompany', 'accomplish', 'account',
       'accountant', 'accuse', 'ace', 'achieve', 'act', 'action',
       'actionhero', 'activist', 'activity', 'actor', 'actress', 'actual',
       'actually', 'adam', 'adambrody', 'adamgoldberg', 'adamsandler',
       'adamscott', 'adamshankman', 'adapt', 'adaptation', 'add',
       'addict', 'addiction', 'adewaleakinnuoye', 'adolescence',
       'adolescent', 'adopt'], 

In [542]:
from sklearn.metrics.pairwise import cosine_similarity
similarity1 = cosine_similarity(vectors1)

In [543]:
similarity1[0]

array([1.        , 0.08006408, 0.08119979, ..., 0.04399059, 0.0270666 ,
       0.        ], shape=(4800,))

In [544]:
def recommend_function1(movie_name):

    movie_index = new_movies[new_movies['title'] == movie_name].index[0]

    movies_distances = similarity1[movie_index]
    movies_distances = list(enumerate(movies_distances))

    top_movies_distances = sorted(
        movies_distances,
        reverse=True,
        key=lambda x: x[1]
    )
    

    for item in top_movies_distances[1:7]:
        index = item[0]
        print(new_movies.iloc[index]['title'])

In [545]:
recommend_function1('Batman')

Batman & Robin
The Dark Knight Rises
Batman Returns
Batman Begins
Batman v Superman: Dawn of Justice
Batman Forever


In [546]:
recommend_function1('Inception')

Duplex
Silver Medalist
The Helix... Loaded
Looper
Cellular
Blackthorn


#### Model 2 -> TF-IDF Without Using Movie Title Column

In [547]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [548]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

In [549]:
tfidf_vectors1 = tfidf.fit_transform(new_movies['tags'])

In [550]:
tfidf_vectors1

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 143708 stored elements and shape (4800, 5000)>

In [551]:
tfidf_similarity1 = cosine_similarity(tfidf_vectors1)

In [552]:
tfidf_similarity1[0]

array([1.        , 0.02050399, 0.02819118, ..., 0.02045123, 0.00614673,
       0.        ], shape=(4800,))

In [553]:
def recommend_function2(movie_name):

    movie_index = new_movies[new_movies['title'] == movie_name].index[0]

    movies_distances = tfidf_similarity1[movie_index]
    movies_distances = list(enumerate(movies_distances))

    top_movies_distances = sorted(
        movies_distances,
        reverse=True,
        key=lambda x: x[1]
    )


    for item in top_movies_distances[1:7]:
        index = item[0]
        print(new_movies.iloc[index]['title'])

In [554]:
recommend_function2('Avatar')

Aliens
Battle: Los Angeles
Falcon Rising
Apollo 18
Star Trek Into Darkness
Ender's Game


In [555]:
recommend_function2('Inception')

The Dark Knight Rises
Don Jon
Cypher
Premium Rush
Pitch Perfect 2
The East


    BOW vs TF-IDF Experiment

    I experimented with both Bag-of-Words and TF-IDF for representing movie tags. Both approaches were able to identify relevant movies for queries such as Batman, Avatar, and The Dark Knight. However, both methods also produced some irrelevant recommendations for movies such as Inception and Titanic.

    The experiments showed that changing the vectorization technique alone does not guarantee better recommendations. The quality of the recommendation also depends heavily on the quality of the movie tags and the features included in them.

    Therefore, instead of selecting a vectorizer only on the basis of cosine similarity scores, I evaluated the actual recommendation results across multiple movies.

    BOW vs TF-IDF: Both approaches produced relevant recommendations for some movies such as Batman and Avatar. However, recommendations for Inception were less semantically consistent. This indicates that the quality of recommendations depends not only on the vectorization technique but also on the quality and composition of the movie tags.

    For Avatar, TF-IDF produced a ranking that aligns more closely with the manually assessed relevance than BOW did.

### Model 3 — BOW With Movie Title  
    Approach:
    
    In the third experiment, the movie title is added to the `tags` representation while continuing to use BOW.
    The purpose is to investigate whether the movie title provides useful additional information for content-based similarity.


In [557]:
new_movies2 =  new_movies[['movie_id','title','tags']]
new_movies2.head()

,movie_id,title,tags
0,19995,Avatar,"22nd century , paraplegic marine dispatch moon..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa , long believe dead , come ba..."
2,206647,Spectre,cryptic message bond ’ past sends trail uncove...
3,49026,The Dark Knight Rises,"follow death district attorney harvey dent , b..."
4,49529,John Carter,"john carter war-weary , former military captai..."


In [558]:
new_movies2['tags'] = new_movies2['tags'] + ' ' + new_movies2['title'].apply(lambda x : x.lower())

In [559]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors2 = cv.fit_transform(new_movies2['tags'])

In [560]:
new_movies2['tags'] = new_movies2['tags'].apply(lemmatize_text)

In [561]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors2 = cv.fit_transform(new_movies2['tags'])

In [562]:
similarity2 = cosine_similarity(vectors2)

In [563]:
similarity2[0]

array([1.        , 0.07509393, 0.08346223, ..., 0.04024218, 0.02418254,
       0.        ], shape=(4800,))

In [564]:
def recommend_function3(movie_name):

    movie_index = new_movies2[new_movies2['title'] == movie_name].index[0]

    movies_distances = similarity2[movie_index]
    movies_distances = list(enumerate(movies_distances))

    top_movies_distances = sorted(
        movies_distances,
        reverse=True,
        key=lambda x: x[1]
    )

    

    for item in top_movies_distances[1:7]:
        index = item[0]
        print(new_movies2.iloc[index]['title'])

In [565]:
recommend_function3('Titanic')

Raise the Titanic
Ghost Ship
The Notebook
Poseidon
The Chambermaid on the Titanic
Under the Same Moon


In [566]:
recommend_function3('Inception')

Duplex
The Helix... Loaded
Looper
Silver Medalist
Cellular
Blackthorn


### Model 4 -> TF-IDF With Using Movie Title Column
Approach:

The fourth experiment combines the two changes
- TF-IDF vectorization
- Movie title included in the `tags` representation
This configuration combines the distinctive-term weighting of TF-IDF with the additional information provided by the movie title.

In [567]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)
tfidf_vectors2 = tfidf.fit_transform(new_movies2['tags'])

In [568]:
tfidf_similarity2 = cosine_similarity(tfidf_vectors2)

In [569]:
def recommend_function4(movie_name):

    movie_index = new_movies2[new_movies2['title'] == movie_name].index[0]

    movies_distances = tfidf_similarity2[movie_index]
    movies_distances = list(enumerate(movies_distances))

    top_movies_distances = sorted(
        movies_distances,
        reverse=True,
        key=lambda x: x[1]
    )
    

    for item in top_movies_distances[1:7]:
        index = item[0]
        print(new_movies2.iloc[index]['title'])


In [570]:
recommend_function4('Titanic')

Raise the Titanic
The Chambermaid on the Titanic
Ghost Ship
Poseidon
Supernova
The Bounty


In [571]:
recommend_function4('Batman Begins')

The Dark Knight
Batman Returns
The Dark Knight Rises
Batman v Superman: Dawn of Justice
Batman
Batman & Robin


### Model comparison

The four configurations were tested using multiple movie queries, including:

- Batman
- Avatar
- Inception
- The Dark Knight
- Titanic

The purpose of this testing was to compare the actual recommendation results rather than selecting a model only from its cosinesimilarity scores.

### Key observations
1. BOW was able to identify movies with overlapping content-related terms.
2. TF-IDF changed the ranking by reducing the influence of commonly occurring words.
3. Including the movie title improved franchise and title-related recommendations.
4. No configuration produced perfect recommendations for every movie.
5. Some movies, such as Inception, still produced mixed recommendations, showing that recommendation quality depends strongly on thequality and representation of the available metadata.
6. Similarity score alone does not guarantee that a recommendation will match human judgement of relevance.

The experiments therefore helped identify the configuration that provided the most suitable behaviour for the current dataset andfeature representation.

In [572]:
movies = [
    'Batman',
    'Avatar',
    'Inception',
    'Titanic'
]

In [573]:
for movie in movies:
    recommend_function1(movie)
    print('-'*100)

Batman & Robin
The Dark Knight Rises
Batman Returns
Batman Begins
Batman v Superman: Dawn of Justice
Batman Forever
----------------------------------------------------------------------------------------------------
Independence Day
Aliens
Titan A.E.
Aliens vs Predator: Requiem
Battle: Los Angeles
Ender's Game
----------------------------------------------------------------------------------------------------
Duplex
Silver Medalist
The Helix... Loaded
Looper
Cellular
Blackthorn
----------------------------------------------------------------------------------------------------
The Notebook
Ghost Ship
Under the Same Moon
Poseidon
Supernova
Captain Phillips
----------------------------------------------------------------------------------------------------


In [574]:
for movie in movies:
    recommend_function2(movie)
    print('-'*100)

Batman & Robin
Batman Returns
The Dark Knight Rises
Batman Begins
Batman v Superman: Dawn of Justice
Batman Forever
----------------------------------------------------------------------------------------------------
Aliens
Battle: Los Angeles
Falcon Rising
Apollo 18
Star Trek Into Darkness
Ender's Game
----------------------------------------------------------------------------------------------------
The Dark Knight Rises
Don Jon
Cypher
Premium Rush
Pitch Perfect 2
The East
----------------------------------------------------------------------------------------------------
Ghost Ship
Supernova
Poseidon
The Bounty
Pirates of the Caribbean: On Stranger Tides
The Black Hole
----------------------------------------------------------------------------------------------------


In [575]:
for movie in movies:
    recommend_function3(movie)
    print('-'*100)

Batman & Robin
The Dark Knight Rises
Batman Returns
Batman Begins
The Dark Knight
Batman Forever
----------------------------------------------------------------------------------------------------
Aliens
Independence Day
Aliens vs Predator: Requiem
Titan A.E.
Small Soldiers
Battle: Los Angeles
----------------------------------------------------------------------------------------------------
Duplex
The Helix... Loaded
Looper
Silver Medalist
Cellular
Blackthorn
----------------------------------------------------------------------------------------------------
Raise the Titanic
Ghost Ship
The Notebook
Poseidon
The Chambermaid on the Titanic
Under the Same Moon
----------------------------------------------------------------------------------------------------


In [576]:
for movie in movies:
    recommend_function4(movie)
    print('-'*100)

Batman Returns
Batman & Robin
The Dark Knight Rises
Batman Begins
The Dark Knight
Batman Forever
----------------------------------------------------------------------------------------------------


Aliens
The Marine
Battle: Los Angeles
Falcon Rising
Apollo 18
Aliens in the Attic
----------------------------------------------------------------------------------------------------
Cypher
Don Jon
Premium Rush
The Dark Knight Rises
Pitch Perfect 2
Looper
----------------------------------------------------------------------------------------------------
Raise the Titanic
The Chambermaid on the Titanic
Ghost Ship
Poseidon
Supernova
The Bounty
----------------------------------------------------------------------------------------------------


# 15. Final Model Selection

Based on the experiments and testing performed above, **TF-IDF with the movie title included in the tags** was selected as the final configuration for the current recommendation system.

### Final Configuration

- Text representation: **TF-IDF**
- Movie title: **Included**
- Main text feature: **`tags`**
- Similarity metric: **Cosine Similarity**
- Text preprocessing: **Lowercasing + Lemmatization**
- Maximum vocabulary size: **5000 features**

The selection was based on the observed recommendation behaviour across multiple movie queries rather than on similarity scores alone.
This configuration will be used for the next stage of the project: building a reusable recommendation pipeline and integrating it into the application.

# 18. Limitations

The current system is a content-based recommendation system, so its recommendations depend heavily on the information available in the movie metadata.

Some limitations include:

- Recommendations depend on the quality of the `overview`, `genres`, `keywords`, `cast` and `crew` information.
- Movies with limited metadata may receive weaker recommendations.
- Cosine similarity measures textual/content similarity but does not directly represent personal user preferences.
- A high similarity score does not always mean that a user will prefer the recommended movie.
- Including movie titles can sometimes increase similarity because of direct word overlap.
- The current system does not use user ratings, watch history or collaborative filtering.

In [579]:
new_movies.to_csv(
    r"C:\Users\moham\Desktop\Movie-Recommender-System\data\processed\movies_processed.csv",
    index=False
)

In [580]:
new_movies.to_csv(
    r"C:\Users\moham\Desktop\Movie-Recommender-System\data\processed\movies_processed_Without_title_In_TagColumn.csv",
    index=False
)